# Fast3R 复现：环境检查

运行下面的代码单元，确认 Notebook 使用的是 `fast3r` Conda 环境，并检查 Python、PyTorch、CUDA、GPU、Fast3R 和关键依赖。

In [2]:
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path


def package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "未安装"


print("=== Python 环境 ===")
print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("Conda 环境:", os.environ.get("CONDA_DEFAULT_ENV", "未检测到"))
print("系统:", platform.platform())

print("\n=== PyTorch / CUDA ===")
try:
    import torch

    print("PyTorch:", torch.__version__)
    print("Torch CUDA:", torch.version.cuda)
    print("CUDA 可用:", torch.cuda.is_available())
    print("GPU 数量:", torch.cuda.device_count())
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(index)
            memory_gb = properties.total_memory / 1024**3
            print(f"GPU {index}: {properties.name} ({memory_gb:.1f} GB)")
except ImportError as error:
    print("PyTorch 导入失败:", error)

print("\n=== Fast3R / 关键依赖 ===")
for package_name in [
    "fast3r",
    "numpy",
    "open3d",
    "jupyterlab",
    "ipykernel",
    "gradio",
    "lightning",
    "huggingface-hub",
]:
    print(f"{package_name}: {package_version(package_name)}")

print("\n=== 项目路径 ===")
project_root = Path.cwd()
print("当前目录:", project_root)
for relative_path in [
    "fast3r",
    "checkpoints/Fast3R_ViT_Large_512",
    "demo_examples",
    "requirements.txt",
]:
    path = project_root / relative_path
    print(f"{relative_path}: {'存在' if path.exists() else '缺失'}")

print("\n=== Fast3R 导入检查 ===")
try:
    import fast3r
    from fast3r.dust3r.inference_multiview import inference
    from fast3r.models.fast3r import Fast3R

    print("Fast3R 导入: 成功")
    print("Fast3R 路径:", fast3r.__file__)
except Exception as error:
    print("Fast3R 导入失败:", repr(error))

=== Python 环境 ===
Python: 3.11.16
解释器: /home/yyz/miniconda3/envs/fast3r/bin/python
Conda 环境: fast3r
系统: Linux-6.8.0-138-generic-x86_64-with-glibc2.35

=== PyTorch / CUDA ===
PyTorch: 2.7.1+cu128
Torch CUDA: 12.8
CUDA 可用: True
GPU 数量: 1
GPU 0: NVIDIA GeForce RTX 5070 Ti Laptop GPU (11.5 GB)

=== Fast3R / 关键依赖 ===
fast3r: 1.0
numpy: 1.26.4
open3d: 0.19.0
jupyterlab: 4.6.3
ipykernel: 7.3.0
gradio: 5.23.0
lightning: 2.6.5
huggingface-hub: 1.30.0

=== 项目路径 ===
当前目录: /home/yyz/fast3r
fast3r: 存在
checkpoints/Fast3R_ViT_Large_512: 存在
demo_examples: 存在
requirements.txt: 存在

=== Fast3R 导入检查 ===
Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead
Fast3R 导入: 成功
Fast3R 路径: /home/yyz/fast3r/fast3r/__init__.py


## 在 Notebook 中启动 Gradio Demo

运行下面的代码单元后，模型和 Gradio 服务会在后台启动。请在浏览器中打开输出的本地地址完成上传和可视化。

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

project_root = Path.cwd()
checkpoint_dir = project_root / "checkpoints" / "Fast3R_ViT_Large_512"
demo_script = project_root / "fast3r" / "viz" / "demo.py"

demo_process = globals().get("demo_process")
if demo_process is not None and demo_process.poll() is None:
    print(f"Demo 已在运行，PID: {demo_process.pid}")
    print("浏览器地址: http://127.0.0.1:7860")
else:
    import torch

    can_start = True
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / 1024**3
        print(f"GPU 可用显存: {free_gb:.2f} / {total_bytes / 1024**3:.2f} GB")
        if free_gb < 4.0:
            can_start = False
            print("暂不启动 Demo：可用显存不足 4 GB。")
            print("请先停止其他 GPU 训练任务，再重新运行本单元。")

    if can_start:
        if not checkpoint_dir.exists():
            raise FileNotFoundError(f"找不到模型目录: {checkpoint_dir}")
        if not demo_script.exists():
            raise FileNotFoundError(f"找不到 Demo 脚本: {demo_script}")

        environment = os.environ.copy()
        environment["GRADIO_TEMP_DIR"] = str(project_root / "gradio_tmp_dir")
        demo_process = subprocess.Popen(
            [
                sys.executable,
                str(demo_script),
                "--checkpoint_dir",
                str(checkpoint_dir),
                "--examples_dir",
                str(project_root / "demo_examples"),
                "--output_dir",
                str(project_root / "demo_outputs"),
            ],
            cwd=project_root,
            env=environment,
            start_new_session=True,
        )
        time.sleep(2)
        if demo_process.poll() is not None:
            raise RuntimeError(f"Demo 启动失败，退出码: {demo_process.returncode}")

        print(f"Demo 已在后台启动，PID: {demo_process.pid}")
        print("请在浏览器打开: http://127.0.0.1:7860")
        print("如果端口被占用，请先运行停止单元，再重新启动。")

GPU 可用显存: 9.65 / 11.47 GB
Demo 已在后台启动，PID: 155946
请在浏览器打开: http://127.0.0.1:7860
如果端口被占用，请先运行停止单元，再重新启动。


Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead


/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/__init__.py:11: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/__init__.py:11: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/lightning_fabric/__init__.py:29: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:34: UnderReviewWarning: The feature generate_power_seq is currently marked under review. The compatibility with other Lightning projects

[13:21:02] ViserServerManager started                                 ]8;id=5371292;file:///home/yyz/fast3r/fast3r/viz/demo.py\demo.py]8;;\:]8;id=5371293;file:///home/yyz/fast3r/fast3r/viz/demo.py#77\77]8;;\
Loading weights from local directory
* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
Extracted 9 frames from video using OpenCV
>> Loading a list of 9 images
 - adding /home/yyz/fast3r/gradio_tmp_dir/fe467f95460a11e742346fca7534409eeaa5b1085c11fa7175c8c045404d2ac8/frame_000000.jpg with resolution 1080x1920 --> 224x224
 - adding /home/yyz/fast3r/gradio_tmp_dir/e4f6520fa4467711fb9627d138d182d584dec1ac74d1a99e1bd9f406a6defb44/frame_000001.jpg with resolution 1080x1920 --> 224x224
 - adding /home/yyz/fast3r/gradio_tmp_dir/445cc20fdba3f402684f0abaad49372758abb7e3756549bf6ce524d22ae91fb2/frame_000002.jpg with resolution 1080x1920 --> 224x224
 - adding /home/yyz/fast3r/gradio_tmp_dir/f962ce1cf64570c236da6bf0d9a4341e10e12acbc

100%|██████████| 9/9 [00:00<00:00, 486.47it/s]


(viser) Share URL requested!
(viser) Generated share URL (expires in 24 hours, max 16 clients): 
https://abstract-visual.share.viser.studio
[13:22:13] Server 1 launched with URL                                ]8;id=5371306;file:///home/yyz/fast3r/fast3r/viz/demo.py\demo.py]8;;\:]8;id=5371307;file:///home/yyz/fast3r/fast3r/viz/demo.py#127\127]8;;\
           https://abstract-visual.share.viser.studio (pid: 157691)             
           for session session_1789536130877                                    
(viser) Connection opened (0, 1 total), 126 persistent messages
(viser) Connection closed (0, 0 total)


## 停止 Notebook 启动的 Demo

在浏览器中完成观察后，运行下面的单元释放模型占用的显存。

In [11]:
demo_process = globals().get("demo_process")
if demo_process is not None and demo_process.poll() is None:
    demo_process.terminate()
    try:
        demo_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        demo_process.kill()
        demo_process.wait()
    print(f"Demo 已停止，PID: {demo_process.pid}")
    demo_process = None
else:
    print("当前没有由本 Notebook 启动的 Demo 进程。")

Demo 已停止，PID: 155946


## 直接运行多视图推理

下面的单元使用本地 Fast3R checkpoint 和仓库中的示例图片完成一次推理。先运行配置和图片加载单元，再运行模型推理单元。

In [3]:
from pathlib import Path

import torch

from fast3r.dust3r.inference_multiview import inference
from fast3r.dust3r.utils.image import load_images
from fast3r.utils.checkpoint_utils import load_model

project_root = Path.cwd()
checkpoint_dir = project_root / "checkpoints" / "Fast3R_ViT_Large_512"
output_dir = project_root / "demo_outputs" / "notebook_inference"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not checkpoint_dir.exists():
    raise FileNotFoundError(f"找不到模型目录: {checkpoint_dir}")

candidate_dirs = [
    project_root / "demo_outputs" / "notebook_family_frames",
    project_root / "demo_outputs" / "example_scenes",
]
image_extensions = {".jpg", ".jpeg", ".png", ".heic", ".heif"}
filelist = []
input_dir = None
for candidate_dir in candidate_dirs:
    candidate_files = sorted(
        path
        for path in candidate_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    if len(candidate_files) >= 2:
        input_dir = candidate_dir
        filelist = candidate_files
        break

if len(filelist) < 2:
    raise RuntimeError(
        "没有找到至少 2 张输入图片。请先准备图片到 "
        "demo_outputs/notebook_family_frames/。"
    )

print("设备:", device)
print("模型目录:", checkpoint_dir)
print("输入目录:", input_dir)
print("示例图片数量:", len(filelist))
print("前 5 张图片:")
for path in filelist[:5]:
    print(" -", path.relative_to(project_root))

/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/__init__.py:11: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/__init__.py:11: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/lightning_fabric/__init__.py:29: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:34: UnderReviewWarning: The feature generate_power_seq is currently marked under review. The compatibility with other Lightning projects

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
设备: cuda
模型目录: /home/yyz/fast3r/checkpoints/Fast3R_ViT_Large_512
输入目录: /home/yyz/fast3r/demo_outputs/notebook_family_frames
示例图片数量: 29
前 5 张图片:
 - demo_outputs/notebook_family_frames/frame_000000.jpg
 - demo_outputs/notebook_family_frames/frame_000001.jpg
 - demo_outputs/notebook_family_frames/frame_000002.jpg
 - demo_outputs/notebook_family_frames/frame_000003.jpg
 - demo_outputs/notebook_family_frames/frame_000004.jpg


In [4]:
max_views = 12
selected_filelist = [str(path) for path in filelist[:max_views]]
images = load_images(selected_filelist, size=512, verbose=True)

print(f"已加载 {len(images)} 个视角")
print("首个视角字段:", sorted(images[0].keys()))
print("首个视角图像形状:", tuple(images[0]["img"].shape))

>> Loading a list of 12 images
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000000.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000001.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000002.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000003.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000004.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000005.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000006.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000007.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/n

In [5]:
print("正在加载 Fast3R 模型...")
model, lit_module = load_model(
    str(checkpoint_dir),
    device=device,
    is_lightning_checkpoint=False,
)
model.eval()
lit_module.eval()

with torch.inference_mode():
    output_dict, profiling_info = inference(
        images,
        model,
        device,
        dtype=torch.float32,
        verbose=True,
        profiling=True,
    )

print("推理完成")
print("模型前向耗时:", profiling_info.get("total_time", "未提供"))
print("输出字段:", sorted(output_dict.keys()))
print("视角数量:", len(output_dict["views"]))
print("预测数量:", len(output_dict["preds"]))

正在加载 Fast3R 模型...
Loading weights from local directory
>> Inference with model on 12 images
encode_images time: 0.5260224342346191
pos emb time: 0.0007810592651367188
decoder time: 0.36101579666137695
head prepare input time: 0.0013949871063232422
head forward time: 0.32763218879699707
total Fast3R forward time: 1.21708083152771
推理完成
模型前向耗时: 1.21708083152771
输出字段: ['loss', 'preds', 'views']
视角数量: 12
预测数量: 12


In [6]:
print("=== 预测结构 ===")
for index, prediction in enumerate(output_dict["preds"][:3]):
    print(f"视角 {index}:")
    print("  字段:", sorted(prediction.keys()))
    for key in ["pts3d_in_other_view", "conf", "pts3d_local", "conf_local"]:
        value = prediction.get(key)
        if value is not None:
            print(f"  {key}: shape={tuple(value.shape)}, dtype={value.dtype}")

first_points = output_dict["preds"][0]["pts3d_in_other_view"]
first_confidence = output_dict["preds"][0]["conf"]
print("首个视角点图数值范围:", float(first_points.min()), float(first_points.max()))
print("首个视角置信度范围:", float(first_confidence.min()), float(first_confidence.max()))

=== 预测结构 ===
视角 0:
  字段: ['conf', 'conf_local', 'pts3d_in_other_view', 'pts3d_local']
  pts3d_in_other_view: shape=(1, 288, 512, 3), dtype=torch.float32
  conf: shape=(1, 288, 512), dtype=torch.float32
  pts3d_local: shape=(1, 288, 512, 3), dtype=torch.float32
  conf_local: shape=(1, 288, 512), dtype=torch.float32
视角 1:
  字段: ['conf', 'conf_local', 'pts3d_in_other_view', 'pts3d_local']
  pts3d_in_other_view: shape=(1, 288, 512, 3), dtype=torch.float32
  conf: shape=(1, 288, 512), dtype=torch.float32
  pts3d_local: shape=(1, 288, 512, 3), dtype=torch.float32
  conf_local: shape=(1, 288, 512), dtype=torch.float32
视角 2:
  字段: ['conf', 'conf_local', 'pts3d_in_other_view', 'pts3d_local']
  pts3d_in_other_view: shape=(1, 288, 512, 3), dtype=torch.float32
  conf: shape=(1, 288, 512), dtype=torch.float32
  pts3d_local: shape=(1, 288, 512, 3), dtype=torch.float32
  conf_local: shape=(1, 288, 512), dtype=torch.float32
首个视角点图数值范围: -0.6300936937332153 1.9896825551986694
首个视角置信度范围: 1.0 36.946483612

In [12]:
poses_c2w_batch, estimated_focals = lit_module.estimate_camera_poses(
    output_dict["preds"],
    niter_PnP=100,
    focal_length_estimation_method="first_view_from_global_head",
)

camera_poses = poses_c2w_batch[0]
focals = estimated_focals[0]
print("相机数量:", len(camera_poses))
print("第一个相机位姿形状:", tuple(camera_poses[0].shape))
print("估计焦距数量:", len(focals))
print("第一个相机位姿:\n", camera_poses[0])
print("估计焦距:", [float(focal) for focal in focals])

相机数量: 12
第一个相机位姿形状: (4, 4)
估计焦距数量: 12
第一个相机位姿:
 [[ 9.9997616e-01 -3.2196178e-03  6.1094072e-03 -6.2514952e-04]
 [ 3.2308705e-03  9.9999315e-01 -1.8328696e-03  8.5467205e-04]
 [-6.1034649e-03  1.8525644e-03  9.9997973e-01 -1.8026977e-04]
 [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  1.0000000e+00]]
估计焦距: [312.47137451171875, 312.47137451171875, 312.47137451171875, 312.47137451171875, 312.47137451171875, 312.47137451171875, 312.47137451171875, 312.47137451171875, 312.47137451171875, 312.47137451171875, 312.47137451171875, 312.47137451171875]


In [13]:
import numpy as np

output_dir.mkdir(parents=True, exist_ok=True)
pointmaps = np.stack(
    [prediction["pts3d_in_other_view"].detach().cpu().numpy().squeeze(0)
     for prediction in output_dict["preds"]]
)
confidence_maps = np.stack(
    [prediction["conf"].detach().cpu().numpy().squeeze(0)
     for prediction in output_dict["preds"]]
)
poses_array = np.stack(
    [pose.detach().cpu().numpy() if hasattr(pose, "detach") else np.asarray(pose)
     for pose in camera_poses]
)
focals_array = np.asarray([float(focal) for focal in focals], dtype=np.float32)

result_path = output_dir / "reconstruction_results.npz"
np.savez_compressed(
    result_path,
    pointmaps=pointmaps,
    confidence_maps=confidence_maps,
    poses_c2w=poses_array,
    estimated_focals=focals_array,
    image_paths=np.asarray(selected_filelist),
)
print("复现结果已保存:", result_path)
print("点图数组:", pointmaps.shape)
print("置信度数组:", confidence_maps.shape)

复现结果已保存: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_results.npz
点图数组: (12, 288, 512, 3)
置信度数组: (12, 288, 512)


## 模块级验证

这一部分使用 forward hooks 观察一次真实推理中各组件的输出，不改变模型计算。每个组件都检查输出是否存在、形状是否合理、数值是否有限。

In [7]:
from collections.abc import Mapping


def tensor_summary(value):
    if torch.is_tensor(value):
        detached = value.detach()
        return {
            "type": "tensor",
            "shape": tuple(detached.shape),
            "dtype": str(detached.dtype),
            "device": str(detached.device),
            "finite": bool(torch.isfinite(detached).all().item()),
            "min": float(detached.min().item()) if detached.numel() else None,
            "max": float(detached.max().item()) if detached.numel() else None,
        }
    if isinstance(value, Mapping):
        return {str(key): tensor_summary(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [tensor_summary(item) for item in value[:3]]
    return {"type": type(value).__name__}


module_outputs = {}


def capture_output(module_name):
    def hook(_module, _inputs, output):
        module_outputs[module_name] = tensor_summary(output)
    return hook


hook_handles = [
    model.encoder.patch_embed.register_forward_hook(capture_output("patch_embed")),
    model.encoder.register_forward_hook(capture_output("encoder")),
    model.decoder.register_forward_hook(capture_output("decoder")),
    model.downstream_head.register_forward_hook(capture_output("global_head")),
]
if model.downstream_head_local is not None:
    hook_handles.append(
        model.downstream_head_local.register_forward_hook(capture_output("local_head"))
    )

hooked_output, hooked_profile = inference(
    images,
    model,
    device,
    dtype=torch.float32,
    verbose=False,
    profiling=True,
)

for handle in hook_handles:
    handle.remove()

required_modules = {"patch_embed", "encoder", "decoder", "global_head"}
missing_modules = required_modules - module_outputs.keys()
if missing_modules:
    raise RuntimeError(f"没有捕获到模块输出: {sorted(missing_modules)}")

for module_name, summary in module_outputs.items():
    print(f"[{module_name}]\n{summary}")

hooked_predictions = hooked_output["preds"]
assert len(hooked_predictions) == len(images)
assert all("pts3d_in_other_view" in prediction for prediction in hooked_predictions)
assert all(torch.isfinite(prediction["pts3d_in_other_view"]).all() for prediction in hooked_predictions)
assert all(torch.isfinite(prediction["conf"]).all() for prediction in hooked_predictions)
print("模块级验证通过")

encode_images time: 0.26081323623657227
pos emb time: 0.0007436275482177734
decoder time: 0.30457305908203125
head prepare input time: 0.00025010108947753906
head forward time: 0.36727261543273926
total Fast3R forward time: 0.93385910987854
[patch_embed]
[{'type': 'tensor', 'shape': (12, 576, 1024), 'dtype': 'torch.float16', 'device': 'cuda:0', 'finite': True, 'min': -13.984375, 'max': 13.3203125}, {'type': 'tensor', 'shape': (12, 576, 2), 'dtype': 'torch.int64', 'device': 'cuda:0', 'finite': True, 'min': 0.0, 'max': 31.0}]
[encoder]
[{'type': 'tensor', 'shape': (12, 576, 1024), 'dtype': 'torch.float32', 'device': 'cuda:0', 'finite': True, 'min': -8.948944091796875, 'max': 10.216102600097656}, {'type': 'tensor', 'shape': (12, 576, 2), 'dtype': 'torch.int64', 'device': 'cuda:0', 'finite': True, 'min': 0.0, 'max': 31.0}]
[decoder]
[{'type': 'tensor', 'shape': (1, 6912, 1024), 'dtype': 'torch.float32', 'device': 'cuda:0', 'finite': True, 'min': -8.948944091796875, 'max': 10.21610260009765

## 视角规模与性能验证

用相同场景的不同视角数量运行推理，记录前向时间和显存峰值。这个实验用于验证模型的多视图输入行为，不等同于论文完整 benchmark。

In [8]:
import gc
import time

benchmark_results = []
for view_count in [4, 8, 12]:
    benchmark_images = images[:view_count]
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
    start_time = time.perf_counter()
    benchmark_output, benchmark_profile = inference(
        benchmark_images,
        model,
        device,
        dtype=torch.float32,
        verbose=False,
        profiling=True,
    )
    elapsed = time.perf_counter() - start_time
    peak_memory_gb = (
        torch.cuda.max_memory_allocated(device) / 1024**3
        if device.type == "cuda"
        else float("nan")
    )
    benchmark_results.append(
        {
            "views": view_count,
            "total_time": float(benchmark_profile["total_time"]),
            "wall_time": elapsed,
            "peak_memory_gb": peak_memory_gb,
            "output_count": len(benchmark_output["preds"]),
        }
    )
    del benchmark_images, benchmark_output, benchmark_profile
    gc.collect()

for result in benchmark_results:
    print(
        f"视角={result['views']:>2} | "
        f"模型时间={result['total_time']:.3f}s | "
        f"墙钟时间={result['wall_time']:.3f}s | "
        f"峰值显存={result['peak_memory_gb']:.2f}GB | "
        f"输出={result['output_count']}"
    )

assert all(result["output_count"] == result["views"] for result in benchmark_results)
print("视角规模验证通过")

encode_images time: 0.08369660377502441
pos emb time: 0.0004687309265136719
decoder time: 0.0719149112701416
head prepare input time: 0.000152587890625
head forward time: 0.07248663902282715
total Fast3R forward time: 0.22889161109924316
encode_images time: 0.15852141380310059
pos emb time: 0.0005595684051513672
decoder time: 0.20783185958862305
head prepare input time: 0.0002689361572265625
head forward time: 0.1598196029663086
total Fast3R forward time: 0.5271825790405273
encode_images time: 0.25908780097961426
pos emb time: 0.0005745887756347656
decoder time: 0.32266902923583984
head prepare input time: 0.00025272369384765625
head forward time: 0.3164489269256592
total Fast3R forward time: 0.8992273807525635
视角= 4 | 模型时间=0.229s | 墙钟时间=0.242s | 峰值显存=4.56GB | 输出=4
视角= 8 | 模型时间=0.527s | 墙钟时间=0.544s | 峰值显存=5.31GB | 输出=8
视角=12 | 模型时间=0.899s | 墙钟时间=0.920s | 峰值显存=6.13GB | 输出=12
视角规模验证通过


## 点云导出与结果检查

使用 global prediction 的点图和置信度生成彩色 PLY。置信度阈值采用全体点的百分位数，避免低置信度区域淹没可视化结果。

In [10]:
import numpy as np
import open3d as o3d

all_points = []
all_colors = []
all_confidences = []

for prediction, view in zip(output_dict["preds"], images):
    points = prediction["pts3d_in_other_view"].numpy().squeeze(0)
    confidence = prediction["conf"].numpy().squeeze(0)
    image_tensor = view["img"].squeeze(0).permute(1, 2, 0).numpy()

    if image_tensor.min() < 0:
        image_tensor = (image_tensor + 1.0) / 2.0
    if image_tensor.max() > 1:
        image_tensor = image_tensor / 255.0
    colors = np.clip(image_tensor, 0.0, 1.0)

    all_points.append(points.reshape(-1, 3))
    all_colors.append(colors.reshape(-1, 3))
    all_confidences.append(confidence.reshape(-1))

all_points = np.concatenate(all_points, axis=0)
all_colors = np.concatenate(all_colors, axis=0)
all_confidences = np.concatenate(all_confidences, axis=0)

confidence_percentile = 45
confidence_threshold = np.percentile(all_confidences, confidence_percentile)
valid = (
    np.isfinite(all_points).all(axis=1)
    & np.isfinite(all_colors).all(axis=1)
    & np.isfinite(all_confidences)
    & (all_confidences >= confidence_threshold)
)

filtered_points = all_points[valid]
filtered_colors = all_colors[valid]
max_points = 1_000_000
if len(filtered_points) > max_points:
    sample_indices = np.linspace(0, len(filtered_points) - 1, max_points, dtype=int)
    filtered_points = filtered_points[sample_indices]
    filtered_colors = filtered_colors[sample_indices]

point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(filtered_points.astype(np.float64))
point_cloud.colors = o3d.utility.Vector3dVector(filtered_colors.astype(np.float64))
ply_path = output_dir / "reconstruction_pointcloud.ply"
written = o3d.io.write_point_cloud(str(ply_path), point_cloud)

if not written:
    raise RuntimeError(f"PLY 写入失败: {ply_path}")
print("置信度阈值:", float(confidence_threshold))
print("过滤后点数:", len(filtered_points))
print("PLY 已保存:", ply_path)
assert ply_path.exists() and ply_path.stat().st_size > 0

置信度阈值: 9.571455001831055
过滤后点数: 973345
PLY 已保存: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_pointcloud.ply


## 复现结果汇总与论文级评估边界

到这里完成的是本地 checkpoint 的组件级和端到端推理验证。论文中的 DTU、7-Scenes、Neural-RGBD 等指标还需要按照项目文档准备预处理数据，并运行 `configs/eval/` 下对应的评估配置。

In [14]:
summary = {
    "environment": (
        os.environ.get("CONDA_DEFAULT_ENV") == "fast3r"
        and device.type == "cuda"
    ),
    "input_views": len(images),
    "predictions": len(output_dict["preds"]),
    "module_hooks": sorted(module_outputs.keys()),
    "pose_count": len(camera_poses),
    "pointcloud_points": len(filtered_points),
    "results_file": str(result_path),
    "ply_file": str(ply_path),
}

print("=== Fast3R 复现汇总 ===")
print("环境检查:", "通过" if summary["environment"] else "请检查")
print("输入/预测视角:", summary["input_views"], "/", summary["predictions"])
print("已捕获模块:", ", ".join(summary["module_hooks"]))
print("相机位姿数量:", summary["pose_count"])
print("PLY 点数:", summary["pointcloud_points"])
print("NPZ:", summary["results_file"])
print("PLY:", summary["ply_file"])

assert summary["environment"]
assert summary["input_views"] == summary["predictions"] == summary["pose_count"]
assert {"patch_embed", "encoder", "decoder", "global_head"}.issubset(
    set(summary["module_hooks"])
)
assert Path(summary["results_file"]).exists()
assert Path(summary["ply_file"]).exists()
print("本地组件级复现全部通过")

=== Fast3R 复现汇总 ===
环境检查: 通过
输入/预测视角: 12 / 12
已捕获模块: decoder, encoder, global_head, local_head, patch_embed
相机位姿数量: 12
PLY 点数: 973345
NPZ: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_results.npz
PLY: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_pointcloud.ply
本地组件级复现全部通过
